# 04 · Anomaly Detection
**Climate Trend Analyzer** — step 4 of 5

### What is an "anomaly" in climate data?
A month whose behaviour is **unusual for that calendar month and that place**.
A 30 °C January in London is an anomaly; a 30 °C July in London is not.

### Four detectors, four blind spots
| Method | Catches | Blind spot |
| --- | --- | --- |
| Climatology z-score | deviations from the calendar-month normal | needs a stable baseline |
| STL residual | deviations from trend + seasonality | depends on the decomposition |
| IQR (Tukey fences) | gross statistical outliers | blunt for skewed data |
| Isolation Forest | unusual *combinations* of variables | unsupervised, needs tuning |

The results are **merged into one anomaly register** with a severity score -
that table is what a city planner or a reinsurer would actually read.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Launch Jupyter from the project root or notebooks directory")
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)

from src import config as cfg
from src import data_loader, preprocessing, features, eda, trend_analysis
from src import anomaly_detection, forecasting, insights, visualizations as viz

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("Project root:", PROJECT_ROOT)

Project root: /home/user/Climate-Trend-Analyzer


In [2]:
feat = pd.read_csv(cfg.FEATURED_CSV, parse_dates=["date"])
anomalies = anomaly_detection.detect_anomalies(feat)
print(f"Total anomalies: {len(anomalies)}")
anomalies.head(10)

Total anomalies: 362


,date,region,variable,method,value,expected,deviation,score,severity,direction,description,anomaly_id
0,1980-02-01,New York,rainfall_mm,climatology_zscore,431.50,114.80,275.70,3.116,moderate,wet,"Wet: 432 mm vs normal 115 mm (376% of normal, ...",ANM-0001
1,1980-05-01,Sydney,temp_mean_c,climatology_zscore,12.09,15.98,-3.89,-2.562,moderate,cold,Cold anomaly: 12.1°C vs normal 16.0°C (-2.6σ),ANM-0002
2,1980-08-01,New York,rainfall_mm,climatology_zscore,357.00,57.10,525.50,4.717,high,wet,"Wet: 357 mm vs normal 57 mm (626% of normal, +...",ANM-0003
3,1980-08-01,New York,multivariate,isolation_forest,25.68,23.67,2.01,0.091,high,hot,"Unusual climate state (temp +2.0°C, rain +526%)",ANM-0004
4,1981-03-01,Sydney,rainfall_mm,climatology_zscore,315.30,89.90,250.60,3.143,moderate,wet,"Wet: 315 mm vs normal 90 mm (351% of normal, +...",ANM-0005
5,1981-06-01,London,temp_mean_c,stl_residual,18.75,15.44,3.31,2.766,moderate,hot,Unexpected 18.8°C month: residual +3.3°C (+2.8σ),ANM-0006
6,1981-08-01,New York,temp_mean_c,stl_residual,20.41,23.80,-3.39,-3.070,high,cold,Unexpected 20.4°C month: residual -3.4°C (-3.1σ),ANM-0007
7,1981-10-01,London,temp_mean_c,climatology_zscore,8.17,11.56,-3.39,-2.826,moderate,cold,Cold anomaly: 8.2°C vs normal 11.6°C (-2.8σ),ANM-0008
8,1981-10-01,London,temp_mean_c,stl_residual,8.17,12.60,-4.43,-3.695,high,cold,Unexpected 8.2°C month: residual -4.4°C (-3.7σ),ANM-0009
9,1981-11-01,Sydney,rainfall_mm,climatology_zscore,315.40,79.20,298.10,3.303,moderate,wet,"Wet: 315 mm vs normal 79 mm (398% of normal, +...",ANM-0010


## 4.1 How many, and by which method?

In [3]:
anomalies["method"].value_counts().to_frame("count")

,count
method,
stl_residual,175
climatology_zscore,114
isolation_forest,54
iqr,19


In [4]:
anomaly_detection.anomaly_summary(anomalies)

,region,total,high,hot,cold
0,Hyderabad,67,22,39,14
1,London,63,16,33,16
2,New York,75,27,34,27
3,Singapore,103,22,46,19
4,Sydney,54,18,25,15


## 4.2 The most extreme events in the record

In [5]:
anomaly_detection.notable_events(anomalies, top_n=12)[
    ["anomaly_id","date","region","variable","value","expected","deviation","score","description"]
]

,anomaly_id,date,region,variable,value,expected,deviation,score,description
320,ANM-0321,2020-01-01,Singapore,temp_mean_c,36.90,29.68,7.22,6.076,Unexpected 36.9°C month: residual +7.2°C (+6.1σ)
80,ANM-0081,1990-02-01,Hyderabad,temp_mean_c,36.96,30.33,6.63,6.071,Unexpected 37.0°C month: residual +6.6°C (+6.1σ)
76,ANM-0077,1990-01-01,Sydney,temp_mean_c,31.70,25.26,6.44,5.808,Unexpected 31.7°C month: residual +6.4°C (+5.8σ)
194,ANM-0195,2005-01-01,London,temp_mean_c,9.17,2.37,6.80,5.675,Unexpected 9.2°C month: residual +6.8°C (+5.7σ)
212,ANM-0213,2005-03-01,New York,temp_mean_c,13.46,7.25,6.21,5.615,Unexpected 13.5°C month: residual +6.2°C (+5.6σ)
85,ANM-0086,1990-02-01,Singapore,temp_mean_c,36.06,29.50,6.56,5.523,Unexpected 36.1°C month: residual +6.6°C (+5.5σ)
334,ANM-0335,2020-02-01,Singapore,temp_mean_c,36.24,29.88,6.36,5.354,Unexpected 36.2°C month: residual +6.4°C (+5.4σ)
186,ANM-0187,2004-01-01,London,temp_mean_c,7.57,1.26,6.31,5.265,Unexpected 7.6°C month: residual +6.3°C (+5.3σ)
198,ANM-0199,2005-01-01,Singapore,temp_mean_c,34.20,28.00,6.20,5.216,Unexpected 34.2°C month: residual +6.2°C (+5.2σ)
202,ANM-0203,2005-02-01,London,temp_mean_c,10.85,4.61,6.24,5.207,Unexpected 10.8°C month: residual +6.2°C (+5.2σ)


## 4.3 Visualising the detections

In [6]:
viz.plot_anomaly_detection(feat, anomalies)
viz.plot_anomaly_timeline(anomalies)

[viz] saved 09_anomaly_detection.png


[viz] saved 10_anomaly_timeline.png


PosixPath('/home/user/Climate-Trend-Analyzer/outputs/figures/10_anomaly_timeline.png')

## 4.4 Extreme events (heat waves and droughts)

In [7]:
viz.plot_extreme_events(feat)

[viz] saved 11_extreme_events_by_decade.png


PosixPath('/home/user/Climate-Trend-Analyzer/outputs/figures/11_extreme_events_by_decade.png')

The heat-wave flag requires **3 consecutive months** above +1.5 σ, and the
drought flag requires rainfall below 40 % of the calendar-month normal -
business definitions, not just statistical ones.

## 4.5 Does the detector find what we planted? (validation)

In [8]:
# The simulator knows where it injected extreme events - check recall
truth = feat[feat["sim_heatwave"] == 1][["date","region"]].drop_duplicates()
detected = set(zip(anomalies[anomalies.direction=="hot"]["date"],
                   anomalies[anomalies.direction=="hot"]["region"]))
hit = sum((d, r) in detected for d, r in zip(truth["date"], truth["region"]))
print(f"Injected heat-wave months : {len(truth)}")
print(f"Also flagged as anomalies  : {hit}")
print(f"Recall                    : {100*hit/max(len(truth),1):.1f}%")

Injected heat-wave months : 45
Also flagged as anomalies  : 44
Recall                    : 97.8%


### ✅ Checkpoint
* 4 detection methods agree on a common register
* Recall against the injected ground truth is measured (not assumed)
* The register is exported to `outputs/tables/anomalies.csv`